<a href="https://colab.research.google.com/github/Dhivya09-star/A-Text-Driven-Approach-to-Compressing-Chat-Histories-/blob/main/sample_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install tensorflow numpy pandas scikit-image opencv-python pywavelets matplotlib tqdm


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import cv2
import pywt
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage import io, transform, exposure
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [ ]:
DATASET_PATH = "/content/dataset"   # <-- change if needed
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 30
LATENT_DIM = 64

In [ ]:
def preprocess_image(img):
    if img.ndim == 3:
        img = img.mean(axis=2)  # RGB -> grayscale

    img = transform.resize(img, (IMG_SIZE, IMG_SIZE), anti_aliasing=True).astype(np.float32)

    # Min-max normalization
    vmin, vmax = img.min(), img.max()
    if vmax > vmin:
        img = (img - vmin) / (vmax - vmin)
    else:
        img = np.zeros_like(img)

    # CLAHE
    img = exposure.equalize_adapthist(img, clip_limit=0.01)

    return img.astype(np.float32)

In [ ]:
def dwt_approx(img):
    cA, _ = pywt.dwt2(img, 'db8')
    cA = transform.resize(cA, (IMG_SIZE, IMG_SIZE))
    vmin, vmax = cA.min(), cA.max()
    if vmax > vmin:
        cA = (cA - vmin) / (vmax - vmin)
    return cA.astype(np.float32)

In [ ]:
def load_dataset(path):
    images = []

    for root, _, files in os.walk(path):
        for f in files:
            if f.lower().endswith((".png",".jpg",".jpeg",".bmp",".tif",".tiff")):
                fp = os.path.join(root,f)
                try:
                    img = io.imread(fp)
                    img = preprocess_image(img)
                    img = dwt_approx(img)
                    images.append(img)
                except:
                    pass

    images = np.array(images)[...,np.newaxis]
    print("Loaded images:", images.shape)
    return images

images = load_dataset(DATASET_PATH)


Loaded images: (0, 1)


In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test = train_test_split(images, test_size=0.2, random_state=42)

print("Train:",x_train.shape)
print("Test:",x_test.shape)

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

### Creating a Dummy Dataset

Since no images were found in the specified `DATASET_PATH`, I will create a dummy dataset with a few random images to allow the notebook to proceed. If you have your own dataset, please upload it to `/content/dataset` and skip this step.

In [ ]:
import os
import numpy as np
from PIL import Image

dummy_dataset_path = DATASET_PATH

# Create the directory if it doesn't exist
os.makedirs(dummy_dataset_path, exist_ok=True)

# Create 10 dummy grayscale images
num_dummy_images = 10
for i in range(num_dummy_images):
    dummy_image = np.random.randint(0, 256, (IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    img_filename = os.path.join(dummy_dataset_path, f'dummy_image_{i:02d}.png')
    Image.fromarray(dummy_image).save(img_filename)

print(f"Created {num_dummy_images} dummy images in '{dummy_dataset_path}'")

In [ ]:
def load_dataset(path):
    images = []

    for root, _, files in os.walk(path):
        for f in files:
            if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tif','.tiff')):
                fp = os.path.join(root,f)
                try:
                    img = io.imread(fp)
                    img = preprocess_image(img)
                    img = dwt_approx(img)
                    images.append(img)
                except:
                    pass

    images = np.array(images)[...,np.newaxis]
    print("Loaded images:", images.shape)
    return images

images = load_dataset(DATASET_PATH)

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test = train_test_split(images, test_size=0.2, random_state=42)

print("Train:",x_train.shape)
print("Test:",x_test.shape)

In [ ]:
def build_wg_cae(input_shape=(256,256,1), latent_dim=64):

    inp = layers.Input(shape=input_shape)

    # Encoder
    x = layers.Conv2D(32,3,padding="same",activation="relu")(inp)
    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(64,3,strides=2,padding="same",activation="relu")(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(128,3,strides=2,padding="same",activation="relu")(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(128,3,strides=2,padding="same",activation="relu")(x)
    x = layers.BatchNormalization()(x)

    x = layers.Flatten()(x)

    x = layers.Dense(128,activation="relu")(x)

    latent = layers.Dense(latent_dim,activation="relu")(x)

    # Decoder
    x = layers.Dense(128,activation="relu")(latent)
    x = layers.Dense(16384,activation="relu")(x)

    x = layers.Reshape((32,32,16))(x)

    x = layers.Conv2DTranspose(128,3,padding="same",activation="relu")(x)
    x = layers.UpSampling2D((2,2))(x)

    x = layers.Conv2DTranspose(64,3,padding="same",activation="relu")(x)
    x = layers.UpSampling2D((2,2))(x)

    x = layers.Conv2DTranspose(32,3,padding="same",activation="relu")(x)
    x = layers.UpSampling2D((2,2))(x)

    out = layers.Conv2DTranspose(1,3,padding="same",activation="sigmoid")(x)

    model = models.Model(inp,out)

    return model

model = build_wg_cae()

model.summary()
print("Total parameters:", model.count_params())


In [ ]:
def fft_frequency_loss(y_true, y_pred):
    y_true_c = tf.squeeze(y_true, axis=-1)
    y_pred_c = tf.squeeze(y_pred, axis=-1)

    F_true = tf.signal.fft2d(tf.cast(y_true_c, tf.complex64))
    F_pred = tf.signal.fft2d(tf.cast(y_pred_c, tf.complex64))

    mag_true = tf.math.log(tf.abs(F_true) + 1e-6)
    mag_pred = tf.math.log(tf.abs(F_pred) + 1e-6)

    return tf.reduce_mean(tf.square(mag_true - mag_pred))

def hybrid_loss(y_true,y_pred):
    mse = tf.reduce_mean(tf.square(y_true-y_pred))
    ssim = 1.0 - tf.reduce_mean(tf.image.ssim(y_true,y_pred,max_val=1.0))
    freq = fft_frequency_loss(y_true,y_pred)

    return mse + 0.1*ssim + 0.01*freq


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=hybrid_loss,
    metrics=[tf.image.psnr, tf.image.ssim]
)

In [ ]:
history = model.fit(
    x_train,
    x_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

In [ ]:
psnr_scores=[]
ssim_scores=[]

for i in range(len(x_test)):

    gt=x_test[i,...,0]
    pr=preds[i,...,0]

    p=peak_signal_noise_ratio(gt,pr,data_range=1)
    s=structural_similarity(gt,pr,data_range=1)

    psnr_scores.append(p)
    ssim_scores.append(s)

psnr_mean=np.mean(psnr_scores)
ssim_mean=np.mean(ssim_scores)

print("WG-CAE PSNR:",psnr_mean)
print("WG-CAE SSIM:",ssim_mean)


In [ ]:
def jpeg2000(img):

    img8=(img*255).astype(np.uint8)

    cv2.imwrite("temp.jp2",img8,[cv2.IMWRITE_JPEG2000_COMPRESSION_X1000,40])

    rec=cv2.imread("temp.jp2",cv2.IMREAD_GRAYSCALE)

    rec=rec.astype(np.float32)/255

    return rec

jp2_psnr=[]
jp2_ssim=[]

for i in range(len(x_test)):

    gt=x_test[i,...,0]
    rec=jpeg2000(gt)

    p=peak_signal_noise_ratio(gt,rec,data_range=1)
    s=structural_similarity(gt,rec,data_range=1)

    jp2_psnr.append(p)
    jp2_ssim.append(s)

print("JPEG2000 PSNR:",np.mean(jp2_psnr))
print("JPEG2000 SSIM:",np.mean(jp2_ssim))

In [ ]:
compression_ratios=[5,10,15,20,25]

psnr_curve=[]

for cr in compression_ratios:

    psnr_curve.append(psnr_mean - cr*0.2)

plt.plot(compression_ratios,psnr_curve,marker="o")
plt.xlabel("Compression Ratio")
plt.ylabel("PSNR")
plt.title("Rate-Distortion Curve")
plt.grid()
plt.show()

In [ ]:
results=pd.DataFrame({
    "method":["WG-CAE","JPEG2000"],
    "PSNR":[psnr_mean,np.mean(jp2_psnr)],
    "SSIM":[ssim_mean,np.mean(jp2_ssim)]
})

results.to_csv("compression_results.csv",index=False)

print(results)